# Reproducing, from scratch: *Social attention spikes precede larger intraday price moves in either direction*

This notebook **re-implements the method of the Instrumetriq research note step by step** - it does not call a
pre-written script. Every quantity is built here in the notebook from the raw dataset fields, so you can see
exactly which field is read, how fields are combined, and the arithmetic behind each number.

It uses only the **free weekly samples** (one Tier 3 day per week). No purchase or private data is required -
we clone the public repository solely to obtain the sample `.parquet` files; all computation happens below.

**The question.** When a coin's count of distinct posting authors spikes to >= 6x its own recent baseline, is a
large intraday price move (a >= +/-4% excursion, either direction) more common in the following ~2-hour session than
on normal-attention sessions? We measure the **lift** (spike-rate / normal-rate) with a day-level bootstrap interval.

**Expected result on the free weekly samples: ~2.4x.** (The full-archive headline uses a multi-day baseline and is
~3-4x; the weekly sample can only support an *intraday* baseline, which is noisier and attenuates the lift - see
the note for the reconciliation.)

## 0. Setup

`pandas` / `numpy` / `pyarrow` are pre-installed on Colab; the install line is just a safeguard.

In [ ]:
!pip -q install pandas pyarrow numpy
import glob
import numpy as np
import pandas as pd

## 1. Get the sample data (only the parquet files)

We clone the public repo to obtain the free weekly samples. This is an anonymous read of a **public** repo - no
login or authorization is needed. Nothing but the `.parquet` files is used from here on.

In [ ]:
!git clone --depth 1 https://github.com/SiCkGFX/instrumetriq-public.git 2>/dev/null || (cd instrumetriq-public && git pull -q)
FILES = sorted(glob.glob('instrumetriq-public/samples/week_*/*_tier3.parquet'))
print(len(FILES), 'weekly Tier 3 sample files')
print(FILES[0], '...', FILES[-1])

## 2. What one record looks like

Each **row** in a Tier 3 parquet is one coin observed over one ~2-hour tracking session. We use exactly two
nested columns:

- `twitter_sentiment_windows.last_cycle` - the social snapshot taken **at session admission** (this is our
  *attention* input, and it is fixed before the price path that follows, so there is no look-ahead).
- `spot_prices` - the list of ~700 price ticks (~10 s apart) that make up the session (our *outcome* input).

Let's print the exact field paths from a single record so there is no ambiguity about provenance.

In [ ]:
df0 = pd.read_parquet(FILES[0], columns=['symbol','snapshot_ts','twitter_sentiment_windows','spot_prices'])
r   = df0.iloc[0]
tsw = r['twitter_sentiment_windows']
lc  = tsw.get('last_cycle') if hasattr(tsw, 'get') else None
authors = (lc.get('author_stats') or {}).get('distinct_authors_total')
posts   = lc.get('posts_total')
silent  = (lc.get('sentiment_activity') or {}).get('is_silent')
sp      = r['spot_prices']

print('symbol                          :', r['symbol'])
print('snapshot_ts                     :', r['snapshot_ts'])
print('distinct_authors_total          :', authors,
      '  <- twitter_sentiment_windows.last_cycle.author_stats.distinct_authors_total')
print('posts_total                     :', posts,
      '  <- twitter_sentiment_windows.last_cycle.posts_total')
print('is_silent                       :', silent,
      '  <- twitter_sentiment_windows.last_cycle.sentiment_activity.is_silent')
print('number of price ticks           :', len(sp))
print('first tick mid (entry price m0) :', sp[0]['mid'], '  <- spot_prices[0].mid')

## 3. Extract the two inputs per session

For every record we keep only sessions with **observable social activity** (`posts_total > 0`, not `is_silent`, a
non-null author count), then compute the forward price move from the price path:

1. Take the **mid** of every **3rd** tick (`spot_prices[::3]` -> ~30 s spacing) and let `m0` be the first.
2. **maxgain** = `max(mid)/m0 - 1` (best upside reached during the session).
3. **mdd** = `min(mid)/m0 - 1` (worst drawdown reached).

`maxgain` and `mdd` are all we need: a *large move* is `maxgain >= +4%` **or** `mdd <= -4%`.
The attention input we keep is `distinct_authors_total`.

In [ ]:
def extract(files):
    rows = []
    for fp in files:
        df = pd.read_parquet(fp, columns=['symbol','snapshot_ts',
                                          'twitter_sentiment_windows','spot_prices'])
        for sym, ts, tsw, sp in zip(df['symbol'].values, df['snapshot_ts'].values,
                                    df['twitter_sentiment_windows'].values, df['spot_prices'].values):
            lc = tsw.get('last_cycle') if hasattr(tsw, 'get') else None
            if lc is None:
                continue
            # --- quality gates: observable social activity, not silent ---
            posts   = lc.get('posts_total')
            silent  = (lc.get('sentiment_activity') or {}).get('is_silent')
            authors = (lc.get('author_stats') or {}).get('distinct_authors_total')
            if not posts or silent or authors is None:
                continue
            # --- forward move from the ~2h price path (every 3rd mid ~= 30s) ---
            if sp is None or len(sp) < 6:
                continue
            mids = [float(s['mid']) for s in sp[::3] if s.get('mid')]
            if len(mids) < 5 or not mids[0]:
                continue
            a = np.asarray(mids); m0 = a[0]
            rows.append((sym, pd.Timestamp(ts), float(authors),
                         a.max()/m0 - 1,   # maxgain
                         a.min()/m0 - 1))  # mdd (<= 0)
    d = pd.DataFrame(rows, columns=['symbol','ts','authors','maxgain','mdd'])
    d['ts']  = pd.to_datetime(d['ts'], utc=True)
    d['day'] = d['ts'].dt.strftime('%Y-%m-%d')
    return d.sort_values(['symbol','ts'])

d = extract(FILES)
print(f'{len(d):,} sessions with observable activity and a valid price path, across {d.symbol.nunique()} coins')
d.head()

## 4. Attention spike = author count / the coin's own recent baseline

Attention is judged **relative to each coin's own history**, not in absolute terms. The *baseline* is a
**past-only median** of the coin's prior sessions - `shift(1)` guarantees the current session is excluded, so
there is no look-ahead.

- On the **full contiguous archive** the baseline is the last **20 sessions across days** (`rolling(20, min_periods=5)`).
- The **weekly samples are one non-contiguous day (Sunday) per week**, so a cross-day trailing baseline is not
  reconstructable. We instead use an **intraday** baseline: each coin versus its *earlier same-day sessions*
  (`groupby([symbol, day]) -> shift(1).expanding(min_periods=5).median()`). This is the same construction, just
  scoped to the day - noisier, which is why the sample lift is attenuated relative to the archive headline.

The **spike ratio** is `authors / baseline`. A session is a **spike** if the ratio >= 6, and **normal** if it is
between 0.8 and 1.2 (sessions in between are compared to neither group).

In [ ]:
# intraday baseline (correct construction for the weekly Sunday samples)
base = d.groupby(['symbol','day'])['authors'].transform(
    lambda s: s.shift(1).expanding(min_periods=5).median())
d = d.assign(baseline=base)
d = d[d['baseline'] > 0].copy()
d['spike'] = d['authors'] / d['baseline']

is_spike  = d['spike'] >= 6.0
is_normal = (d['spike'] >= 0.8) & (d['spike'] <= 1.2)
print(f'spike sessions  (>=6x baseline): {int(is_spike.sum()):,}')
print(f'normal sessions (0.8-1.2x)     : {int(is_normal.sum()):,}')

## 5. Large move (either direction) and the lift

A session shows a **large move** if `maxgain >= +4%` **or** `mdd <= -4%` - an excursion of at least 4% in *either*
direction. The **rate** in a group is the fraction of its sessions with a large move; the **lift** is the spike
rate divided by the normal rate.

In [ ]:
large_move = (d['maxgain'] >= 0.04) | (d['mdd'] <= -0.04)

rate_spike  = large_move[is_spike].mean()
rate_normal = large_move[is_normal].mean()
lift = rate_spike / rate_normal

print(f'large-move rate  spike : {100*rate_spike:.1f}%')
print(f'large-move rate  normal: {100*rate_normal:.1f}%')
print(f'LIFT (spike / normal)  : {lift:.2f}x')

## 6. Uncertainty: a day-level block bootstrap

Coin-sessions are **not independent**: a market-wide attention surge lights up many coins on the same day - that
is *one* event, not hundreds. Resampling individual sessions would treat them as independent and understate the
uncertainty. So we resample **whole calendar days** with replacement (1,000 times), recompute the lift from the
pooled sessions of the resampled days each time, and take the 2.5th-97.5th percentiles as a 95% interval.

We pre-aggregate per day (spike count, spike-with-move count, normal count, normal-with-move count) so each
bootstrap draw is a fast sum.

In [ ]:
work = d.assign(_s=is_spike.values, _n=is_normal.values, _h=large_move.values)
work = work.assign(_sh=work['_s'] & work['_h'], _nh=work['_n'] & work['_h'])
agg = work.groupby('day').agg(ns=('_s','sum'), nsh=('_sh','sum'),
                              nn=('_n','sum'), nnh=('_nh','sum'))
ns, nsh, nn, nnh = (agg[c].values.astype(float) for c in ('ns','nsh','nn','nnh'))

D = len(agg); rng = np.random.default_rng(0); lifts = []
for _ in range(1000):
    i = rng.integers(0, D, D)                 # resample days with replacement
    S, SH, N, NH = ns[i].sum(), nsh[i].sum(), nn[i].sum(), nnh[i].sum()
    if S and N and NH:
        lifts.append((SH/S) / (NH/N))
lo, hi = np.percentile(lifts, 2.5), np.percentile(lifts, 97.5)
print(f'LIFT: {lift:.2f}x   [95% day-block bootstrap: {lo:.2f} - {hi:.2f}]')

## 7. Result

The `LIFT` above is the ratio of the large-move rate on attention-spike sessions to the rate on normal-attention
sessions. An interval entirely above 1 indicates spike sessions are associated with larger moves. On the free
weekly samples this reproduces at **~2.4x**.

**Every input was named and every step was arithmetic:** attention from
`twitter_sentiment_windows.last_cycle.author_stats.distinct_authors_total`, the price path from `spot_prices[].mid`;
a past-only median baseline; a +/-4% either-direction move; a day-clustered bootstrap.

**Going further.** A buyer of the full Tier 3 archive reproduces the multi-day headline by swapping the
intraday baseline in Section 4 for the trailing one - `groupby('symbol') -> shift(1).rolling(20, min_periods=5).median()`
- and, for the per-regime figures, restricting to each regime window (see the note's Section 3-Section 4).

Full method, results, robustness, and limitations: the research note in this repository
(`research/attention_volatility.md`).